In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

name = "discrete_val02_tol12"
traj = pd.read_csv(f"{name}_history.csv")
traj_con = traj[traj["setting"] == "con_all"]
traj_uncon = traj[traj["setting"] == "uncon_all"]

final = pd.read_csv(f"{name}.csv")
final["shift"] = (final["test_error"] - final["train_error"])
final_con = final[final["setting"] == "con_all"]
final_uncon = final[final["setting"] == "uncon_all"]


In [ ]:
f, axs = plt.subplots(nrows=10, ncols=4, figsize=(20, 20), sharex=True)

tcg = traj_con.groupby(by=["target", "epoch"]).mean(numeric_only=True)
tug = traj_uncon.groupby(by=["target", "epoch"]).mean(numeric_only=True)

fcg = final_con.groupby(by=["target"]).mean(numeric_only=True)
fug = final_uncon.groupby(by=["target"]).mean(numeric_only=True)

for i, (ax_unl, ax_cl, ax_cd, ax_cv) in enumerate(axs):
    if i == 0:
        ax_unl.set_title("Loss (Unconstrained)")
        ax_cl.set_title("Loss (Constrained)")
        ax_cd.set_title("Mean/max dual")
        ax_cv.set_title("Mean violation")

    ax_unl.plot(tug.loc[f"X{i}"]["selection_loss"])
    ax_cl.plot(tcg.loc[f"X{i}"]["selection_loss"])
    ax_unl.sharey(ax_cl)

    ax_cd.plot(tcg.loc[f"X{i}"]["dual_mean"])
    ax_cd.plot(tcg.loc[f"X{i}"]["dual_max"])
    ax_cv.plot(tcg.loc[f"X{i}"]["violation_mean"])

In [ ]:
(fug - fcg).mean()

In [ ]:
fug["shift"] - fcg["shift"]

In [ ]:
pd.DataFrame({
    "unconstrained shift": fug["shift"],
    "constrained shift": fcg["shift"],
    "shift diff": fug["shift"] - fcg["shift"],
    "unconstrained violation (train)": fug["violation_train"],
    "unconstrained violation (test)": fug["violation_test"],
    "constrained violation (train)": fcg["violation_train"],
    "constrained violation (test)": fcg["violation_test"]
})

In [ ]:
print("unconstrained")
fug

In [ ]:
print("constrained")
fcg

In [ ]:
pd.concat([fug, fcg])[["train_error", "test_error", "violation_train", "violation_test", "shift"]].corr()